### **smFISH data processing used for Fig. 1 and Extended Data Fig. 1**

##### This notebook provides an example of how STARR-FISH data was processed from sequential FISH-based imaging. This pipeline applies to both the enhancer and T7 transcript imaging.

#### **Analysis Overview:**

1. Spot fitting
2. Spot thresholding
3. Cell segmentation
4. Cell-by-gene matrix generation

#### **Key outputs:**
- Fitted spot data and thresholding parameters based primarily on spot brightness
- Cell segmentation masks
- A cell-by-gene matrix for enhancer activities (transcript counts)

#### **1. Spot fitting**

##### *1a. Map the data paths and variables used for fitting*

In [ ]:
# All functions used for image processing can be found in ioMicro.py
from ioMicro import *

# Define data folder where image files (.zarr) are stored
data_folder = r'Z:\zgibbs\raw_data\20CRE'

# Map all the hybes (rounds of imaging)
hybes =  glob.glob(data_folder+os.sep+'H*')

# Map all the fovs (fields of view)
fovs = [os.path.basename(fl)for fl in glob.glob(hybes[0]+os.sep+'*.zarr')]
def get_Hi(fld): 
    try: return int(os.path.basename(fld)[1:]) 
    except: return -1
hybes = np.array(hybes)[np.argsort([get_Hi(hybe) for hybe in hybes ])]

##### *1b. Define function used for spot fitting*

In [ ]:
def main_do_compute_fits(save_folder,hybe,fov,icol,save_fl,psf,old_method):
    im_ = read_im(hybe+os.sep+fov)
    im__ = np.array(im_[icol],dtype=np.float32)
    
    if old_method:
        im_n = norm_slice(im__,s=30)
        Xh = get_local_maxfast_tensor(im_n,th_fit=500,im_raw=im__,dic_psf=None,delta=1,delta_fit=3,sigmaZ=1,sigmaXY=1.5,gpu=True)
    else:
        fl_med = flat_field_tag+'med_col_raw'+str(icol)+'.npz'
        if os.path.exists(fl_med):
            im_med = np.array(np.load(fl_med)['im'],dtype=np.float32)
            im_med = cv2.blur(im_med,(20,20))
            im__ = im__/im_med*np.median(im_med)
        else:
            print("Did not find flat field")
        try:
            Xh = get_local_max_tile(im__,th=3600,s_ = 500,pad=100,psf=psf,plt_val=None,snorm=30,gpu=True,
                                    deconv={'method':'wiener','beta':0.0001},
                                    delta=1,delta_fit=3,sigmaZ=1,sigmaXY=1.5)
        except:
            Xh = get_local_max_tile(im__,th=3600,s_ = 500,pad=100,psf=psf,plt_val=None,snorm=30,gpu=False,
                                    deconv={'method':'wiener','beta':0.0001},
                                    delta=1,delta_fit=3,sigmaZ=1,sigmaXY=1.5)
    np.savez_compressed(save_fl,Xh=Xh)

##### *1c. Compute and save the fits*

In [ ]:
save_folder = r'Z:\zgibbs\raw_data\20CRE\enhancer_fits'
tag = os.path.basename(hybe)
save_fl = data_folder+os.sep+fov.split('.')[0]+'--'+tag+'_fits_icol'+str(icol)+'.npz'
psf = r'C:\Scripts\NMERFISH\psfs\psf_750_Scope2.npy'

for fov in tqdm(fovs):
    for hybe in hybes[1:]:
        for icol in range(3):
            main_do_compute_fits(save_folder,hybe,fov,icol,save_fl,psf,old_method=True)

#### **2. Spot thresholding**

##### *2a. Load the raw images into Napari to view and draw layers of polygons around cells with positive or negative signal*

In [ ]:
from dask import array as da
import napari

fls = glob.glob(r'Z:\zgibbs\raw_data\20CRE\H*\Conv_zscan__126.zarr') # example image files for one fov across all hybes
ims = []
for fl in fls[1:-1]:
    ims.append(read_im(fl)[np.newaxis])
ims = da.concatenate(ims)

V = napari.view_image(ims)

##### *2b. Select two shape layers of polygons, one for positive (clear signal) cells and another for background (lack of spots) cells*

In [ ]:
# Load in the coordinates of the masks you drew
coords_neg = [X[:,-2:]for X in V.layers[-1].data] # most recent layer, keep track of the order you drew them in
coords_pos = [X[:,-2:]for X in V.layers[-2].data]

##### *2c. Load in a reference hybe, then define positive and negative points (using 1 color channel)*

In [ ]:
from shapely import Polygon,Point

# specific to this fov and color channel, assumption is that hybe shouldn't matter so much but you can test others to be sure
fov = 126
CRE = np.load(rf'Z:\zgibbs\raw_data\20CRE\enhancer_fits\Conv_zscan__{fov:03d}--H{5}_fits_icol{1}.npz')['Xh']

pols_pos = [Polygon(X) for X in coords_pos]
pols_neg = [Polygon(X) for X in coords_neg]

vol_pos = np.sum([p.area for p in pols_pos])
vol_neg = np.sum([p.area for p in pols_neg])

CREXY = CRE[:,1:3]

is_positive_cell = [np.any([pol.contains(Point(x)) for pol in pols_pos]) for x in CREXY]
is_negative_cell = [np.any([pol.contains(Point(x)) for pol in pols_neg]) for x in CREXY]

##### *2d. Plot brightness value distrubutions from spots inside positive or negative cells for this fov & color channel*

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

ax.hist(np.log(CRE[is_positive_cell][:,-1]),density=True,alpha=0.5, color='g', label = 'positive cells')
ax.hist(np.log(CRE[is_negative_cell][:,-1]),density=True,alpha=0.5, label = 'negative cells')
ax.set_xlabel('Brightness')
ax.set_ylabel('Frequency')
ax.legend()

##### *2e. Choose threshold where negative cell signal drops off for this color channel*

In [ ]:
# Choose a threshold where the "contamination rate" is ~5-10% or lower, OR based off lowest contamination rate possible, based on plot above
th = np.exp(6.8)
density_pos = np.sum(CRE[is_positive_cell][:,-1]>th)/vol_pos
density_neg = np.sum(CRE[is_negative_cell][:,-1]>th)/vol_neg
print("contamination rate:",density_neg/density_pos)
print(th)

##### *2f. Build list of thresholds (ths), with the best value for each color channel & hybe (each represents an enhancer/T7 species here)*

In [ ]:
# Repeat from Step 2c for all hybe and color channel combinations
ths = [121.5,148.4,181.2,
      221.4,812.4,445.9,
      244.7,244.7,221.4,
      330.3,270.4,221.4,
      221.4,897.8,200.3,
      148.5,270.4,148.4,
      221.4,270.4,148.4]

##### *2g. Save final threshold parameter file*

In [ ]:
Hs = [1,2,3,4,5,6,7]
icols = [0,1,2]

params = [(H,icol) for H in Hs for icol in icols]
final_params = [(H,icol,th)for th,(H,icol) in zip(ths,params)]

np.savez('enhancer_final_params_ths.npz',final_params=final_params)

#### **3. Cell segmentation**

##### *3a. Configure variables for segmentation*

In [ ]:
from ioMicro import read_im
import numpy as np
import glob
import zarr
import os

experiment = '20CRE'
segname = 'polyA'
data_folder = r'Z:\zgibbs\raw_data\20CRE'
save_dir = rf'C:\Users\zgibbs\cellpose\{experiment}\\{segname}'

##### *3b. Identify and count files (fovs) to segment*

In [ ]:
zarr_files = r'Z:\zgibbs\raw_data\20CRE\H5\Conv_zscan__*.zarr'
files = list(sorted(list(glob.glob(zarr_files))))
print(f"Found {len(files)} files to segment")

##### *3c. Read image data from all FOVs to be used for segmentation*

In [ ]:
im_path = r'Z:\zgibbs\raw_data\20CRE\H5\*'
imgs = [read_im(im_path) for im_path in files]

##### *3d. Specify which slice from the image files to use (e.g., dapi or polyA)*

In [ ]:
# Here we extract the polyA channel (channel 3) at the z-slice with the best focus (slice 25)

for i in range(225):
    imgs[i] = imgs[i][3,25,:,:]

##### *3e. Run Cellpose for the image files*

In [ ]:
from cellpose import models, io
from tqdm import tqdm

# Here a pretrained model is used, but you can also use the built-in models
model = models.CellposeModel(gpu=True, pretrained_model=r'C:\Users\zgibbs\cellpose\HCT116_polyA',
                            device=torch.device(0))

# Set the diameter to '0' if using a pretrained model
masks, flows, styles = [], [], []
for img in tqdm(imgs, desc='Segmented FOVs'):
    mask, flow, style = model.eval(img, channels=[0,0], diameter=0)
    masks.append(mask)
    flows.append(flow)
    styles.append(style)

##### *3f. Expand the segmentation masks and remove small masks (debris/artifacts)*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.segmentation import expand_labels

newmasks = []

for mask in masks:
    newmask = mask.copy()
    for cellid, size in zip(*np.unique(mask, return_counts=True)):
        if size < 2500:
            newmask[newmask==cellid] = 0
    newmask = expand_labels(newmask, 3)
    newmasks.append(newmask)

fig, ax = plt.subplots(figsize=(10,10))

ax.imshow(imgs[50], cmap='gray')
ax.contour(newmasks[50], [0.5+x for x in np.unique(newmasks[50])], colors='b')

##### *3g. Save the segmentation masks*

In [ ]:
os.makedirs(save_dir)
savefiles = [os.path.join(save_dir, os.path.basename(file)) for file in files]

io.masks_flows_to_seg(imgs, newmasks, flows, file_names=savefiles, diams=0)
io.save_to_png(imgs, newmasks, flows, file_names=savefiles)

#### **4. Cell-by-gene matrix generation**

##### *4a. Load in th parameter file for enhancer/T7 spots*

In [ ]:
import numpy as np
import pandas as pd

h_ths = np.load('enhancer_final_params_ths.npz')['final_params']
h_ths = h_ths.astype(int)

# create an empty dataframe with hybes (rows) and ths for each icol (columns)
master_ths = pd.DataFrame(index=range(1,8), columns=h_ths[:3,1])

# fill in the dataframe with the correct ths for each hybe/icol
for h in range(7):
    i1,i2 = h*3,(h+1)*3
    master_ths.iloc[h,:] = h_ths[int(i1):int(i2),2]

##### *4b. Create dictionary for all enhancer/T7 spots across FOVs*

In [ ]:
CRE_counts = {}

for fov in tqdm(range(225)):
    for hybe in range(1,8):
        for icol in range(3):
            # load in all fitted spots for this particular image stack
            spots = np.load(rf'Z:\zgibbs\raw_data\20CRE\enhancer_fits\Conv_zscan__{fov:03d}--H{hybe:01d}_fits_icol{icol:01d}.npz')['Xh']
            
            # correct the drift for spots as they're loaded in
            drifts = np.load(rf'Z:\zgibbs\raw_data\20CRE\drifts_polyA\Conv_zscan__{fov:03d}_drift.pkl', allow_pickle=True)
            drift = drifts[f'H{hybe:01d}'][0]
            spots[:,:3]-=drift
            
            # threshold spots based on brightness values from master_ths
            h = spots[:,-1]
            keep = np.where(h > master_ths.iloc[hybe-1, icol])
            spots_ = spots[keep]
            
            # update the dictionary to contain thresholded spots for this particular image stack
            CRE_counts[f'{fov:03d}-H{hybe}-{icol}'] = spots_

##### *4c. Convert keys from CRE_counts into the actual CRE names*

In [ ]:
# make list of CRE names and filter the converted dict keys into a new list, fov_counts

CREs = ('CRE001', 'CRE002', 'CRE003', 'CRE004', 'CRE005', 'CRE006', 'CRE007', 'CRE008',
        'CRE009', 'CRE010', 'CRE011', 'CRE012', 'CRE013', 'CRE014', 'CRE015', 'CRE016',
        'CRE017', 'CRE018', 'CRE019', 'CRE020')

fov_filter = [f'{fov:03d}-H1-0', f'{fov:03d}-H1-1', f'{fov:03d}-H1-2', f'{fov:03d}-H2-0', f'{fov:03d}-H2-1', f'{fov:03d}-H2-2', f'{fov:03d}-H3-0', f'{fov:03d}-H3-1', f'{fov:03d}-H3-2', f'{fov:03d}-H4-0',
              f'{fov:03d}-H4-1', f'{fov:03d}-H4-2', f'{fov:03d}-H5-0', f'{fov:03d}-H5-1', f'{fov:03d}-H5-2', f'{fov:03d}-H6-0', f'{fov:03d}-H6-1', f'{fov:03d}-H6-2', f'{fov:03d}-H7-0', f'{fov:03d}-H7-1']

filterByKey = lambda keys: {ccre_conversion[x[3:]] : CRE_counts[x] for x in keys}

ccre_conversion = {channel_id[3:] : ccre for channel_id, ccre in zip(fov_filter, CREs)}

fov_counts = []

for fov in tqdm(range(225)):
    fov_counts.append(filterByKey(fov_filter))

##### *4d. Create a cell-by-gene matrix for all enhancer/T7 spots using the appropriate segmentation masks*

In [ ]:
import pandas as pd
import numpy as np

masked_spots = []
all_cbg = []
fov_means = []

for fov in tqdm(range(225)):
    # load cellpose masks and add (FOV x 1000) to the mask values to distinguish them from other FOVs
    seg_mask = np.load(rf'C:\Users\zgibbs\cellpose\20CRE\20CRE_seg\polyA\Conv_zscan__{fov:03d}_seg.npy', allow_pickle=True)[()]
    mask_id = fov * 1000 
    seg_mask['masks'] = seg_mask['masks'] + mask_id 

    # specify the appropriate FOV for spots
    spots = fov_counts[fov]
    
    # filter spots to remove those outside the image dimensions following drift correction
    filtered_spots = {key: coords[(coords[:, 2] <= 2999) & (coords[:, 2] > 0) & (coords[:, 1] <= 2999) & (coords[:, 1] > 0), :] for key, coords in spots.items()}
    
    # round the x,y coordinates for decoded spots and assign to the masks from cellpose
    spot_masks = []
    for key in filtered_spots.keys():
        filtered_spots[key][:,2] = np.around(filtered_spots[key][:,2])
        filtered_spots[key][:,1] = np.around(filtered_spots[key][:,1])
        masks = seg_mask['masks'][np.around(filtered_spots[key][:,1]).astype(int), np.around(filtered_spots[key][:,2]).astype(int)]
        spot_masks += [np.array([masks, np.array([key] * (filtered_spots[key].shape[0])), np.around(filtered_spots[key][:,2]), np.around(filtered_spots[key][:,1])])]
        
    spot_masks = np.hstack(spot_masks).T
    barcodes = pd.DataFrame(data=spot_masks, columns=['masks', 'cre_id', 'x_round', 'y_round'])
    cell_by_gene = pd.crosstab(barcodes.masks, barcodes.cre_id)
    fov_means.append(cell_by_gene.mean(axis=0))
    all_cbg.append(cell_by_gene)
    
all_cbg = pd.concat(all_cbg, axis=0)

##### *4e. Remove non-cell masks from the final cell-by-gene matrix*

In [ ]:
non_cells = []

for fov in range(225):
    mask_id = fov * 1000
    non_cells.append(str(mask_id))
    
all_cbg.drop(labels=non_cells, axis=0, inplace = True)

##### *4f. Add new columns to indicate total transcripts & the fov identity for each cell*

In [ ]:
all_cbg['total transcripts'] = all_cbg.sum(axis=1)
all_cbg['fov'] = pd.cut(all_cbg.index.astype(int), bins = np.arange(0, 226)*1000, right=False)

##### *4g. Save the cell-by-gene matrix*

In [ ]:
all_cbg.to_csv(r'C:\Users\zgibbs\analysis\cell_by_gene\SFv4_T7_July_enhancer_cbg.csv')